# Artifacts and evidence lineage: deterministic example

Construct a semantic ID from synthetic derivation inputs and compare it with the hash of a saved payload.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from artifacts.identity import artifact_id, file_sha256

inputs = {"schema": "example-v1", "rows": 2}
same_inputs = {"rows": 2, "schema": "example-v1"}
identity = artifact_id("synthetic-ledger", inputs)
assert identity == artifact_id("synthetic-ledger", same_inputs)
assert identity != artifact_id("synthetic-ledger", {**inputs, "rows": 3})

with TemporaryDirectory() as directory:
    payload = Path(directory) / "ledger.csv"
    payload.write_bytes(b"position,net_pnl\nA,4\nB,-1\n")
    original_digest = file_sha256(payload)
    lineage = {"artifact_id": str(identity), "sha256": original_digest}
    assert file_sha256(payload) == lineage["sha256"]

    payload.write_bytes(b"position,net_pnl\nA,4\nB,-2\n")
    assert file_sha256(payload) != lineage["sha256"]
    assert str(artifact_id("synthetic-ledger", inputs)) == lineage["artifact_id"]

print("Semantic ID:", lineage["artifact_id"])
print("Saved payload digest:", lineage["sha256"])
print("A subsequent payload change was detected.")

Semantic ID: synthetic-ledger:eab5febfa5965fb1a29e073f19f3fc713180e5f846e6da0a37ae87a65dce157e
Saved payload digest: 52288b86f7633f58c48692c066431b33e1292cac383ea565adef554cbb3e9530
A subsequent payload change was detected.


The fixture keeps the semantic inputs fixed while changing a saved byte. A complete claim also needs the derivation, run, parent and period records.